# Colab Runner — grille scope x mécanisme (Full/Selective FT vs LoRA)

**Notebook principal de l'expérimentation.** Il ne contient aucune logique métier :
il récupère la dernière version du code (`git clone`/`pull`), puis pilote
`src/forensic_fr` via des paramètres modifiables ci-dessous (cellule *Paramètres*).

Marche à suivre :
1. `Runtime > Change runtime type` → GPU
2. Exécuter les cellules **Récupération du code** et **Environnement** une fois
3. Ajuster la cellule **Paramètres** selon le run voulu, puis l'exécuter
4. Exécuter **Lancement**

Pour relancer un autre volet (ex. passer d'un run unique à la grille complète, ou
activer le contrôle sans ancrage), il suffit de rééditer la cellule *Paramètres* et
de ré-exécuter à partir de là — pas besoin de toucher au code source.

## 1. Récupération du code

`git clone` si le dépôt n'existe pas encore dans la session, `git pull` sinon — donc
toujours la dernière version poussée sur GitHub, y compris en relançant cette cellule
dans une session déjà démarrée.

Pour un dépôt privé : ajouter un secret Colab nommé `GITHUB_TOKEN` (icône clé dans la
barre latérale gauche) plutôt que de coller un token en clair dans `REPO_URL` — ce
notebook est versionné sur GitHub, tout ce qui est écrit ici est public.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/MarouaneAyech/foresynch_b2.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/foresynch_b2"  #@param {type:"string"}

try:
    from google.colab import userdata
    _token = userdata.get("GITHUB_TOKEN")
except Exception:
    _token = None

clone_url = REPO_URL
if _token and REPO_URL.startswith("https://"):
    clone_url = REPO_URL.replace("https://", f"https://{_token}@")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Depot deja present dans {REPO_DIR} -> fetch + checkout + pull ({BRANCH})")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
else:
    print(f"Clonage de {REPO_URL} (branche {BRANCH}) dans {REPO_DIR}")
    subprocess.run(["git", "clone", "-b", BRANCH, clone_url, REPO_DIR], check=True)

import sys
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("src ajoute au PYTHONPATH :", os.path.join(REPO_DIR, "src"))

## 2. Environnement

Montage de Drive (données SCface, caches, checkpoints, historiques — voir la table
des dépendances du notebook d'origine) et installation des dépendances.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

FORENSIC_FR_ROOT = "/content/drive/MyDrive/research/forensic_fr"  #@param {type:"string"}
os.environ["FORENSIC_FR_ROOT"] = FORENSIC_FR_ROOT

!pip install -q -r {REPO_DIR}/src/requirements.txt
!pip install -q insightface onnxruntime-gpu

from forensic_fr import config as cfg
cfg.bootstrap_configs(REPO_DIR)  # copie configs/phase1_finetune/*.json du depot vers Drive si absent

## 3. Paramètres de l'expérimentation

C'est la seule cellule à modifier pour changer de volet. `RUN_MODE` bascule entre un
run unique (debug, un contrôle ciblé) et la grille complète ou un sous-ensemble.

Exemples de sous-ensembles utiles (`GRID_ONLY`) :
- `ft_34,full_ft,lora_34` → les 3 configs prioritaires de la Phase 1 (analyse spectrale)
- `ft_34` avec `ANCHOR` décoché → contrôle FT-sans-ancrage (Phase 4.3.2.C du plan)
- vide → grille complète (11 cellules x seeds = jusqu'à 33 runs)

- `fc_only` avec `FREEZE_BN` coché → baseline de contrôle fc seule (Phase 3.1)

Pour le contrôle BN de la Phase 3.2 (déclenché par la Section 7 ci-dessous si une
dérive est détectée) : `RUN_MODE="single"`, `MODE="lora_34"`, `FREEZE_BN` coché.

In [ ]:
#@markdown ### Volet à exécuter
RUN_MODE = "single"  #@param ["single", "grid"]

#@markdown ### Paramètres communs (s'appliquent en mode single ET grid)
EXPERIMENT_ID = "E5_LoRA"  #@param {type:"string"}
N_EPOCHS = 20  #@param {type:"integer"}
ANCHOR = True  #@param {type:"boolean"}
FREEZE_BN = False  #@param {type:"boolean"}

#@markdown ### Si RUN_MODE = "single"
MODE = "lora_34"  #@param ["full_ft", "ft_34", "ft_4", "full_lora", "lora_34", "lora_4", "fc_only", "hybrid"]
SEED = 42  #@param {type:"integer"}
LORA_R = 8  #@param {type:"integer"}
LORA_ALPHA = 16  #@param {type:"integer"}

#@markdown ### Si RUN_MODE = "grid" — laisser GRID_ONLY vide = toute la grille
GRID_ONLY = ""  #@param {type:"string"}
GRID_SEEDS = "7,42,123"  #@param {type:"string"}

## 4. Lancement

In [ ]:
from forensic_fr.training import RunConfig, build_plan, run_training

if RUN_MODE == "single":
    rc = RunConfig(
        mode=MODE, seed=SEED, lora_r=LORA_R, lora_alpha=LORA_ALPHA,
        anchor=ANCHOR, freeze_bn=FREEZE_BN, n_epochs=N_EPOCHS, experiment_id=EXPERIMENT_ID,
    )
    result = run_training(rc)
    print("Checkpoint :", result["checkpoint_path"])
    print("Historique :", result["history_path"])

elif RUN_MODE == "grid":
    only = [m.strip() for m in GRID_ONLY.split(",") if m.strip()] or None
    seeds = [int(s.strip()) for s in GRID_SEEDS.split(",") if s.strip()]
    plan = build_plan(only=only, seeds=seeds, anchor=ANCHOR, freeze_bn=FREEZE_BN,
                       n_epochs=N_EPOCHS, experiment_id=EXPERIMENT_ID)

    print(f"{len(plan)} run(s) planifies :")
    for rc in plan:
        print(f"  - {rc.mode} r={rc.lora_r} seed={rc.seed} anchor={rc.anchor} freeze_bn={rc.freeze_bn}")

    for i, rc in enumerate(plan, 1):
        print(f"\n[{i}/{len(plan)}] === {rc.mode} seed={rc.seed} ===")
        run_training(rc)

else:
    raise ValueError(f"RUN_MODE inconnu : {RUN_MODE!r}")

## 5. Résultats produits

Checkpoints (`.pt`) et historiques (`exp1_{mode}.json`) du run (Drive, persistant
entre sessions).

In [ ]:
from forensic_fr import config as cfg

run_dir = cfg.run_dir(EXPERIMENT_ID)

print("Checkpoints :")
for p in sorted((run_dir / "checkpoints").glob("*.pt")):
    print(" -", p)

print("\nHistoriques :")
for p in sorted(run_dir.glob("exp1_*.json")):
    print(" -", p)

## 6. Analyse spectrale (Phase 1)

Charge les checkpoints produits ci-dessus, calcule le rang effectif de Delta W
par couche (SVD) pour les configs FT, et reconstruit le Delta W effectif de LoRA
`(alpha/r)*B*A` pour comparaison. Aucune donnée d'entraînement requise — juste les
checkpoints déjà sur Drive. Sorties : `analysis/outputs/spectral_per_layer.csv` et
les spectres bruts dans `analysis/outputs/spectra/`.

In [ ]:
!python {REPO_DIR}/analysis/scripts/spectral_analysis.py --experiment-id {EXPERIMENT_ID}

## 7. Contrôle BatchNorm (Phase 1.4) ⛔ obligatoire

`model.train()` est actif pendant tout l'entraînement — un BatchNorm met à jour ses
statistiques courantes (`running_mean`/`running_var`) dès qu'il voit des données en
mode train, **même si ses paramètres affines sont gelés** (`requires_grad=False` ne
bloque que le gradient, pas les stats courantes). Ce contrôle vérifie si le backbone
LoRA est réellement figé, ou si ces statistiques dérivent — un confondant potentiel
pour la comparaison FT vs LoRA en infrarouge (où les statistiques d'image sont très
différentes du visible).

In [ ]:
!python {REPO_DIR}/analysis/scripts/bn_drift_check.py --experiment-id {EXPERIMENT_ID}

## 8. Figure des spectres (Phase 1.2)

Charge un checkpoint `ft_34`, calcule le spectre normalisé de Delta W par couche,
l'agrège par étage (médiane), et trace les 3 courbes en échelle log-log avec des
lignes verticales aux rangs testés dans l'ablation (8/16/32/64). Écrit
`analysis/figures/fig_spectrum.pdf` et affiche en console la fraction d'énergie de
la mise à jour FT capturée à chaque rang — le chiffre à citer dans le texte
("LoRA r=X captures only Y% of the fine-tuning update energy").

In [ ]:
!python {REPO_DIR}/analysis/scripts/spectrum_figure.py --experiment-id {EXPERIMENT_ID}

## 9. Déplacement par étage (Phase 1.3) — clôt la Phase 1

Sur le checkpoint `full_ft` uniquement (le seul scénario où tout le backbone est
libre de bouger), calcule `||W_ft - W_0||_F / ||W_0||_F` agrégé sur **tous les
paramètres** de chaque étage (stem, layer1, layer2, layer3, layer4, fc — pas
seulement les convs 3×3 comme en Phase 1.1). Si le fine-tuning, laissé libre,
concentre spontanément son déplacement sur layer3+4, H1 devient une explication
structurelle, pas seulement une observation du tableau de résultats. Écrit
`analysis/figures/fig_stage_displacement.pdf`.

In [ ]:
!python {REPO_DIR}/analysis/scripts/stage_displacement.py --experiment-id {EXPERIMENT_ID}

## 10. Évaluation complète des checkpoints (Phase 2.0) — prérequis de toute la Phase 2

L'entraînement ne sauvegarde que le rank-1 par terrain. Ce script recharge chaque
checkpoint (réinjection LoRA automatique), recalcule l'évaluation et sauvegarde ce qui
manque pour la suite : embeddings galerie/probes, matrices de scores par terrain, et le
vecteur correct/incorrect par probe avec le rang de la bonne identité. Vérifie que le
rank-1 recalculé est **identique** au JSON d'historique du run.

Sorties dans `runs/E5_LoRA/eval/` : `eval_{run}.npz` (embeddings + scores) et
`correct_{run}.csv` (une ligne par probe). À copier en local pour le bootstrap (2.3),
l'écart d'embedding (2.2) et les CMC (2.1).

`EVAL_ONLY` : préfixes de checkpoints, séparés par des virgules (vide = tous). Le
pré-entraîné (baseline) est toujours évalué.


In [ ]:
EVAL_ONLY = ""  #@param {type:"string"}

_sel = [o.strip() for o in EVAL_ONLY.split(",") if o.strip()]
_only = ("--only " + " ".join(_sel)) if _sel else "--all"
!python {REPO_DIR}/analysis/scripts/eval_checkpoint.py --experiment-id {EXPERIMENT_ID} --pretrained {_only}
